# Computation of the hCF for the Hansen-Zuazua system

The system equations read

$$
\ddot x_1(z,t) = x_1^{''}(z,t), \quad
\ddot x_2(z,t) = x_2^{''}(z,t)
$$

with the boundary conditions

$$
x_1(0,t) = x_2(0,t) = \xi(t),\quad
M \ddot x_1(0,t) = -x_1'(0,t) + x_2'(0,t)
$$

as well as


$$
x_1'(-l_1,t) = -u_1(t),\quad
x_2'(l_2,t) = u_2(t).
$$


## Input parameterisation

In [ ]:
import sympy as sp

# symbols
(l1, l2, mass, z, t, tau) = sp.symbols("l_1 l_2 M z t tau")
y1, y2, u1, u2 = [func(t) for func in sp.symbols("y_1 y_2 u_1 u_2", cls=sp.Function)]
# operators
d_sym, s_sym = sp.symbols("d_t sigma", positive=True)

# shorthands
a1 = b1 = d_sym**2 + 2*d_sym
a2 = b2 = d_sym**2 - 2*d_sym
u_vec = sp.Matrix([u1, u2])
y_vec = sp.Matrix([y1, y2])

# general IO matrix
g_mat = sp.Rational(1,4) * sp.Matrix([
    [a1 * s_sym**l1 + a2 * s_sym**(-l1), -s_sym**l1 -s_sym**(-l1)], 
    [b1 * s_sym**l2 + b2 * s_sym**(-l2), s_sym**l2 +s_sym**(-l2)]])

# substitute actual values
g_mat = g_mat.subs([(l1, sp.pi), (l2, 10), (mass, 1)])

print("Parameterisation of the systen input by the flat output:")
display(sp.Eq(u_vec, sp.MatMul(g_mat, y_vec)))

## System analysis

We use the leading column coefficient matrices to get an overview.
As can be seen, the shifts in the second row of the first column are to big.

In [ ]:
import chimera as ch

print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(g_mat, d_sym))
print("LCCM for shifts:")
display(ch.get_lccm(g_mat, s_sym))
print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(g_mat, d_sym), s_sym)
display(lccm)
print("Is G column-reduced w.r.t. shifts?")
ch.is_col_reduced(g_mat, s_sym)

## Shift reduction

We employ the developed algorithm to reduce the shifts


In [ ]:
gb_mat, l_mat = ch.reduce_matrix_shifts(g_mat, d_sym, s_sym)

print("Computed transformation:")
display(l_mat)
print("Transformed matrix:")
display(gb_mat)

In [ ]:
print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(gb_mat, d_sym))

print("LCCM for shifts:")
display(ch.get_lccm(gb_mat, s_sym))

print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(gb_mat, d_sym), s_sym)
display(lccm)

print("Is the transformed G column-reduced w.r.t. derivatives and shifts?")
print(ch.is_col_reduced(gb_mat, s_sym))

print("How do the reduced shift intervals look like?")
funcs = y1, y2
shift_intervals = []
(lc, ld), (tc, td) = ch.get_matrix_terms(gb_mat, s_sym)
for idx, func in enumerate(funcs):
    shift_intervals.append((min(td[:,idx]), max(ld[:, idx])))
print(funcs)
print(shift_intervals)

print("How do the integrator chain lengths look like?")
derivative_orders = []
(lc, ld), _ = ch.get_matrix_terms(gb_mat, d_sym)
for idx, func in enumerate(funcs):
    derivative_orders.append(max(ld[:, idx]))
print(funcs)
print(derivative_orders)

## Stating the controller form

* Apply inverse Laplace transform
* Simplify convolution integrals with compact support
* Substitute values that are not part of the new state
* Introduce controller form state

In [ ]:
display(gb_mat)
conv_mat = gb_mat.applyfunc(lambda x: ch.inverse_laplace_transform(x, d_sym, s_sym, t))
# display(conv_mat)

print("input")
u_expr = sp.Integral(l_mat.inv().subs(t,tau) @ u_vec.subs(t, t-tau), (tau, -sp.oo, sp.oo)).doit()
display(u_expr)
u_expr = ch.simplify_conv_integrals(u_expr)
display(u_expr)

print("flat output")
y_expr = sp.Integral(conv_mat.subs(t,tau) @ y_vec.subs(t, t-tau), (tau, -sp.oo, sp.oo)).doit()
display(y_expr)
y_expr = ch.simplify_conv_integrals(y_expr)
display(y_expr)
for i in range(len(funcs)):
    y_expr = ch.substitute_non_state_terms(y_expr, funcs[i], derivative_orders[i], *shift_intervals[i])
display(y_expr)